In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import polars as pl
from preprocess_dataset_for_training import create_training_data
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
import matplotlib.pyplot as plt
import pandas as pd
import copy
import re

In [2]:
# -------------------------------
# Reproducibility setup
# -------------------------------
seed = 42  
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
# torch.cuda.manual_seed_all(seed)  # if using multi-GPU

# For deterministic behavior (slower but exact reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# -------------------------------
# Device configuration
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# -------------------------------
# Load and preprocess data
# -------------------------------
files = ["joined_df_with_weather", "joined_df", "joined_df_with_temp"]
for file in files:
    joined_df = pl.read_parquet(f"./data/{file}.parquet")
    X, y = create_training_data(joined_df)
    X_np = X.to_numpy()
    y_np = y.to_numpy()  # shape: (samples, 15 targets)
    del joined_df

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, shuffle=False)
    del X_np, y_np


    scaler_X = StandardScaler()
    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)


    # Target scaling (important for stable float32 regression)
    scaler_y = StandardScaler()
    y_train = scaler_y.fit_transform(y_train)
    y_test = scaler_y.transform(y_test)

    # Convert to PyTorch tensors on CPU (we'll move batches to GPU)
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32)
    del X_train, X_test, y_train, y_test

    # -------------------------------
    # DataLoaders for batching
    # -------------------------------
    batch_size = 256  
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

    test_dataset = TensorDataset(X_test_t, y_test_t)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    feature_names = X.columns 
    lag_groups = {}

    # --- PASS 1: find all bases that have lagged versions -----------------
    lagged_bases = set()

    for fname in feature_names:
        m = re.match(r"(.+)_t.*", fname)
        if m:
            lagged_bases.add(m.group(1))

    # --- PASS 2: populate groups only for bases in lagged_bases -----------
    for i, fname in enumerate(feature_names):
        m = re.match(r"(.+)_t.*", fname)
        if m:
            base = m.group(1)
        else:
            base = fname

        if base in lagged_bases:        # only include if matching lagged version exists
            lag_groups.setdefault(base, []).append(i)


    # -------------------------------
    # Define multi-output NN with per-group stats
    # -------------------------------
    class MultiOutputNN(nn.Module):
        def __init__(self, input_dim, output_dim, lag_groups):
            super().__init__()
            self.lag_groups = list(lag_groups.values())

            self.model = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Linear(256, 256),
                nn.ReLU(),
                nn.Linear(256, output_dim)
            )

        def forward(self, x):
                return self.model(x)
                
        

    input_dim = X_train_t.shape[1]
    output_dim = y_train_t.shape[1]

    def smape(y_true, y_pred, eps=1e-8):
        denom = (np.abs(y_true) + np.abs(y_pred)) + eps
        return 100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom, axis=0)

    num_seeds = 100
    num_targets = output_dim
    batch_size = 256

    # containers
    mae_all = np.zeros((num_seeds, num_targets))
    rmse_all = np.zeros((num_seeds, num_targets))
    r2_all = np.zeros((num_seeds, num_targets))
    smape_all = np.zeros((num_seeds, num_targets))

    # -------------------------------
    # Training loop (unchanged)
    # -------------------------------
    for s in range(num_seeds):
        seed = s
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)

        model_s = MultiOutputNN(input_dim, output_dim, lag_groups).to(device)
        optimizer_s = optim.Adam(model_s.parameters(), lr=1e-4)
        criterion_s = nn.HuberLoss(delta=1.0, reduction='mean')

        # reproducible shuffling for DataLoader
        gen = torch.Generator()
        gen.manual_seed(seed)
        train_loader_s = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=gen)
        test_loader_s = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # early stopping
        best_val_loss = float('inf')
        counter = 0
        patience = 5
        best_state = copy.deepcopy(model_s.state_dict())

        epochs = 100
        for epoch in range(1, epochs + 1):
            model_s.train()
            running_loss = 0.0
            for X_batch, y_batch in train_loader_s:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                optimizer_s.zero_grad()
                outputs = model_s(X_batch)
                loss = criterion_s(outputs, y_batch)
                loss.backward()
                optimizer_s.step()

                running_loss += loss.item() * X_batch.size(0)
            train_loss = running_loss / len(train_loader_s.dataset)

            # validation
            model_s.eval()
            val_loss_total = 0.0
            with torch.no_grad():
                for X_batch, y_batch in test_loader_s:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    outputs = model_s(X_batch)
                    val_loss_total += criterion_s(outputs, y_batch).item() * X_batch.size(0)
            val_loss = val_loss_total / len(test_loader_s.dataset)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                counter = 0
                best_state = copy.deepcopy(model_s.state_dict())
            else:
                counter += 1
                if counter >= patience:
                    break

        # load best model and evaluate
        model_s.load_state_dict(best_state)
        model_s.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for X_batch, y_batch in test_loader_s:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                outputs = model_s(X_batch)
                y_pred_list.append(outputs.cpu())
                y_true_list.append(y_batch.cpu())

        y_pred_scaled = torch.cat(y_pred_list).numpy()
        y_true_scaled = torch.cat(y_true_list).numpy()

        # inverse transform
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        y_true = scaler_y.inverse_transform(y_true_scaled)

        # compute metrics per target
        for i in range(num_targets):
            mae_all[s, i] = mean_absolute_error(y_true[:, i], y_pred[:, i])
            rmse_all[s, i] = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
            r2_all[s, i] = r2_score(y_true[:, i], y_pred[:, i])
        smape_all[s, :] = smape(y_true, y_pred)

    # aggregate: mean across seeds (per-target)
    mae_mean_per_target = mae_all.mean(axis=0)
    rmse_mean_per_target = rmse_all.mean(axis=0)
    r2_mean_per_target = r2_all.mean(axis=0)
    smape_mean_per_target = smape_all.mean(axis=0)

    # also overall averages across targets
    overall = {
        "MAE": mae_mean_per_target.mean(),
        "RMSE": rmse_mean_per_target.mean(),
        "R2": r2_mean_per_target.mean(),
        "SMAPE": smape_mean_per_target.mean()
    }

    # prepare DataFrame per target
    target_names = list(y.columns) if hasattr(y, "columns") else [f"target_{i}" for i in range(num_targets)]
    metrics_df = pd.DataFrame({
        "Target": target_names,
        "MAE": mae_mean_per_target,
        "RMSE": rmse_mean_per_target,
        "R2": r2_mean_per_target,
        "SMAPE": smape_mean_per_target
    })

    display(metrics_df)
    print("Overall averages across targets (mean over seeds then targets):")
    print(overall)

    # optional: save
    metrics_df.to_csv(f"NN_{num_seeds}_seeds_Huber_{file}_12-1.csv", index=False)


,Target,MAE,RMSE,R2,SMAPE
0,Negative Balancing Energy Unit Price for Balan...,37.609325,69.236945,0.484821,112.162462
1,Positive Balancing Energy Unit Price for Balan...,37.656777,69.148406,0.486144,112.221174
2,System Direction (kWh)_t+15min,11368.919707,16599.537369,0.633051,108.003143
3,Negative Balancing Energy Unit Price for Balan...,43.391166,82.431248,0.269836,119.220547
4,Positive Balancing Energy Unit Price for Balan...,43.513315,82.347476,0.271320,119.385757
5,System Direction (kWh)_t+30min,13059.308340,19374.629969,0.500395,116.403007
6,Negative Balancing Energy Unit Price for Balan...,46.308258,87.330212,0.180487,123.698146
7,Positive Balancing Energy Unit Price for Balan...,46.309507,87.229284,0.182379,123.717104
8,System Direction (kWh)_t+45min,14101.975801,21218.870077,0.400841,121.199918
9,Negative Balancing Energy Unit Price for Balan...,48.207037,89.491587,0.139435,126.882469


Overall averages across targets (mean over seeds then targets):
{'MAE': np.float64(4633.848155225118), 'RMSE': np.float64(6995.323360366454), 'R2': np.float64(0.2929535322984059), 'SMAPE': np.float64(121.64098299153646)}


,Target,MAE,RMSE,R2,SMAPE
0,Negative Balancing Energy Unit Price for Balan...,37.531563,69.183165,0.485635,112.101544
1,Positive Balancing Energy Unit Price for Balan...,37.562715,69.048786,0.487635,112.186858
2,System Direction (kWh)_t+15min,11358.620654,16629.180889,0.631800,108.572508
3,Negative Balancing Energy Unit Price for Balan...,43.420808,82.493988,0.268727,119.229922
4,Positive Balancing Energy Unit Price for Balan...,43.485735,82.371337,0.270906,119.378606
5,System Direction (kWh)_t+30min,13055.120527,19400.091526,0.499105,116.915221
6,Negative Balancing Energy Unit Price for Balan...,46.381574,87.404997,0.179086,123.600460
7,Positive Balancing Energy Unit Price for Balan...,46.371211,87.309710,0.180879,123.665694
8,System Direction (kWh)_t+45min,14089.207988,21280.427163,0.397353,122.043955
9,Negative Balancing Energy Unit Price for Balan...,48.347680,89.516476,0.138955,126.783366


Overall averages across targets (mean over seeds then targets):
{'MAE': np.float64(4627.866720324198), 'RMSE': np.float64(7010.983081296927), 'R2': np.float64(0.29180014558633166), 'SMAPE': np.float64(121.85992083231606)}


,Target,MAE,RMSE,R2,SMAPE
0,Negative Balancing Energy Unit Price for Balan...,37.644320,69.132530,0.486396,112.054577
1,Positive Balancing Energy Unit Price for Balan...,37.631796,69.030181,0.487923,112.053207
2,System Direction (kWh)_t+15min,11319.245400,16544.674267,0.635519,107.714415
3,Negative Balancing Energy Unit Price for Balan...,43.467569,82.375390,0.270823,119.156114
4,Positive Balancing Energy Unit Price for Balan...,43.451055,82.248522,0.273070,119.201558
5,System Direction (kWh)_t+30min,13023.441660,19327.003137,0.502871,116.150509
6,Negative Balancing Energy Unit Price for Balan...,46.344156,87.272635,0.181561,123.582714
7,Positive Balancing Energy Unit Price for Balan...,46.350229,87.155712,0.183760,123.643202
8,System Direction (kWh)_t+45min,14066.380117,21186.940168,0.402660,121.226933
9,Negative Balancing Energy Unit Price for Balan...,48.247963,89.418333,0.140847,126.672816


Overall averages across targets (mean over seeds then targets):
{'MAE': np.float64(4623.11621857961), 'RMSE': np.float64(6983.992775087599), 'R2': np.float64(0.29442586612701416), 'SMAPE': np.float64(121.50166135660808)}


In [ ]:
# compute std across seeds (per-target)
mae_mean_per_target = mae_all.mean(axis=0)
mae_std_per_target = mae_all.std(axis=0)

rmse_mean_per_target = rmse_all.mean(axis=0)
rmse_std_per_target = rmse_all.std(axis=0)

r2_mean_per_target = r2_all.mean(axis=0)
r2_std_per_target = r2_all.std(axis=0)

smape_mean_per_target = smape_all.mean(axis=0)
smape_std_per_target = smape_all.std(axis=0)

# add std columns to existing metrics_df (keeps previous mean columns)
metrics_df["MAE_STD"] = mae_std_per_target
metrics_df["RMSE_STD"] = rmse_std_per_target
metrics_df["R2_STD"] = r2_std_per_target
metrics_df["SMAPE_STD"] = smape_std_per_target

# display table and print concise mean ± std per target
display(metrics_df)

for i, t in enumerate(target_names):
    print(
        f"{t}: MAE {mae_mean_per_target[i]:.3f} ± {mae_std_per_target[i]:.3f}, "
        f"RMSE {rmse_mean_per_target[i]:.3f} ± {rmse_std_per_target[i]:.3f}, "
        f"R2 {r2_mean_per_target[i]:.3f} ± {r2_std_per_target[i]:.3f}, "
        f"SMAPE {smape_mean_per_target[i]:.3f} ± {smape_std_per_target[i]:.3f}"
    )